# INSTRUCTOR SOLUTION: Docker Containerization for AI Models
## AIAT 125 — Unit 4: Containerization and Orchestration

**⚠️ INSTRUCTOR USE ONLY — Do not distribute to students**

| Task | Points |
|---|---|
| Task 1: Dockerfile generator | 30 |
| Task 2: Dockerfile parser and validator | 25 |
| Task 3: Container isolation simulation | 25 |
| Task 4: Complete deployment file bundle | 20 |

In [ ]:
import os, json, textwrap

EXERCISE_DIR = "/tmp/docker_exercise"
os.makedirs(EXERCISE_DIR, exist_ok=True)
print("Setup complete.")

---
## Task 1 — Dockerfile Generator (30 points)

In [ ]:
def generate_dockerfile(model_filename, port=8000, python_version="3.10", extra_packages=None):
    """
    Generate a Dockerfile string for serving a model.
    """
    lines = []

    # SOLUTION: FROM instruction
    lines.append(f"FROM python:{python_version}-slim")
    lines.append("")
    lines.append("WORKDIR /app")
    lines.append("")
    lines.append("COPY requirements.txt .")
    lines.append("RUN pip install --no-cache-dir -r requirements.txt")
    lines.append("")

    # SOLUTION: Extra packages before COPY of model/app
    if extra_packages:
        lines.append(f"RUN pip install {' '.join(extra_packages)}")
        lines.append("")

    lines.append(f"COPY {model_filename} /app/")
    lines.append("COPY app.py /app/")
    lines.append("")
    lines.append(f"EXPOSE {port}")
    lines.append("")
    lines.append('CMD ["python", "app.py"]')

    return "\n".join(lines) + "\n"


# --- Generate and print example Dockerfiles ---
df1 = generate_dockerfile("iris_rf.joblib", port=8000)
print("=== Dockerfile (basic) ===")
print(df1)

df2 = generate_dockerfile("sentiment_model.joblib", port=5000,
                           python_version="3.11",
                           extra_packages=["torch", "transformers"])
print("\n=== Dockerfile (with extra packages) ===")
print(df2)

In [ ]:
# Validation: Task 1
assert df1 is not None and isinstance(df1, str)
for instruction in ["FROM python:3.10-slim", "WORKDIR /app",
                    "COPY requirements.txt", "iris_rf.joblib",
                    "EXPOSE 8000", 'CMD ["python", "app.py"]']:
    assert instruction in df1, f"Dockerfile missing: {instruction}"
assert "torch" in df2 and "transformers" in df2
assert "3.11" in df2
print("Task 1 PASSED")

---
## Task 2 — Dockerfile Parser and Validator (25 points)

In [ ]:
def parse_dockerfile(dockerfile_str):
    """
    Count instruction types in a Dockerfile string.
    Returns dict: {instruction_name: count}
    """
    tracked = ["FROM", "WORKDIR", "COPY", "RUN", "EXPOSE", "CMD"]
    counts = {instr: 0 for instr in tracked}

    for line in dockerfile_str.splitlines():
        stripped = line.strip()
        for instr in tracked:
            if stripped.upper().startswith(instr + " ") or stripped.upper() == instr:
                counts[instr] += 1
                break

    return counts


def validate_dockerfile(dockerfile_str):
    """
    Validate a Dockerfile and return a list of issue strings.
    Returns empty list if valid.
    """
    counts = parse_dockerfile(dockerfile_str)
    issues = []

    if counts["FROM"] != 1:
        issues.append(f"Must have exactly 1 FROM, found {counts['FROM']}")
    if counts["WORKDIR"] != 1:
        issues.append(f"Must have exactly 1 WORKDIR, found {counts['WORKDIR']}")
    if counts["CMD"] != 1:
        issues.append(f"Must have exactly 1 CMD, found {counts['CMD']}")
    if counts["COPY"] < 1:
        issues.append("Must have at least 1 COPY")
    if counts["RUN"] < 1:
        issues.append("Must have at least 1 RUN")
    if counts["EXPOSE"] != 1:
        issues.append(f"Must have exactly 1 EXPOSE, found {counts['EXPOSE']}")

    return issues


counts1 = parse_dockerfile(df1)
print("Instruction counts (basic Dockerfile):")
for instr, count in counts1.items():
    print(f"  {instr}: {count}")

issues1 = validate_dockerfile(df1)
if issues1:
    print(f"\nValidation issues: {issues1}")
else:
    print("\nValidation: No issues found")

broken_df = "COPY app.py /app/\nRUN pip install flask\n"
issues_broken = validate_dockerfile(broken_df)
print(f"\nBroken Dockerfile issues: {issues_broken}")

# Validation
assert counts1 is not None and isinstance(counts1, dict)
assert counts1.get("FROM") == 1, f"Expected 1 FROM, got {counts1.get('FROM')}"
assert counts1.get("CMD") == 1, f"Expected 1 CMD, got {counts1.get('CMD')}"
assert issues1 == [], f"Valid Dockerfile should have no issues, got: {issues1}"
assert len(issues_broken) >= 2, f"Broken Dockerfile should have ≥2 issues, got: {issues_broken}"
print("\nTask 2 PASSED")

---
## Task 3 — Container Isolation Simulation (25 points)

In [ ]:
def create_container(name, image, env, packages, port, host_port):
    """
    Create a container configuration dict.
    """
    return {
        "name": name,
        "image": image,
        "env": env,
        "packages": packages,
        "port": port,
        "host_port": host_port,
    }


def check_port_conflict(containers):
    """
    Check if any two containers share the same host_port.
    Returns list of conflicting host_port values.
    """
    host_ports = [c["host_port"] for c in containers]
    seen = set()
    conflicts = []
    for port in host_ports:
        if host_ports.count(port) > 1 and port not in conflicts:
            conflicts.append(port)
    return conflicts


c1 = create_container(
    name="iris-api-v1", image="python:3.10-slim",
    env={"MODEL_VERSION": "1.0", "PORT": "8000"},
    packages={"scikit-learn": "1.3.0", "flask": "3.0.0", "joblib": "1.3.2"},
    port=8000, host_port=8001
)
c2 = create_container(
    name="iris-api-v2", image="python:3.11-slim",
    env={"MODEL_VERSION": "2.0", "PORT": "8000"},
    packages={"scikit-learn": "1.4.0", "flask": "3.0.0", "joblib": "1.3.2"},
    port=8000, host_port=8002
)
c3 = create_container(
    name="iris-api-broken", image="python:3.9-slim",
    env={"MODEL_VERSION": "1.5"},
    packages={"scikit-learn": "1.2.0"},
    port=8000, host_port=8001  # CONFLICT with c1!
)

print("Container 1:", json.dumps(c1, indent=2))
print(f"\nContainer 2 sklearn version: {c2['packages']['scikit-learn']}")
print("Both containers use internal port 8000, mapped to different host ports.")

conflicts_ok = check_port_conflict([c1, c2])
conflicts_bad = check_port_conflict([c1, c2, c3])
print(f"\nConflicts (c1, c2)      : {conflicts_ok}  ← expected []")
print(f"Conflicts (c1, c2, c3)  : {conflicts_bad}  ← expected [8001]")

# Validation
assert c1 is not None and isinstance(c1, dict)
assert c1["name"] == "iris-api-v1"
assert c1["packages"]["scikit-learn"] == "1.3.0"
assert c2["packages"]["scikit-learn"] == "1.4.0"
assert conflicts_ok == [], f"No conflict expected, got {conflicts_ok}"
assert 8001 in conflicts_bad, f"Port 8001 conflict expected, got {conflicts_bad}"
print("\nTask 3 PASSED")

---
## Task 4 — Complete Deployment File Bundle (20 points)

In [ ]:
# SOLUTION 4a: Write Dockerfile
dockerfile_content = generate_dockerfile("iris_rf.joblib", port=8000)
with open(os.path.join(EXERCISE_DIR, "Dockerfile"), "w") as f:
    f.write(dockerfile_content)

# SOLUTION 4b: Write requirements.txt
requirements = "flask==3.0.0\nscikit-learn==1.4.0\njoblib==1.3.2\nnumpy==1.26.0\n"
with open(os.path.join(EXERCISE_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

# SOLUTION 4c: Write app.py
app_code = '''from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load("/app/iris_rf.joblib")
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

@app.get("/health")
def health():
    return jsonify({"status": "ok", "model_loaded": True})

@app.post("/predict")
def predict():
    data = request.get_json(silent=True)
    if not data:
        return jsonify({"error": "JSON body required"}), 400
    try:
        features = [float(data[k]) for k in
                    ["sepal_length", "sepal_width", "petal_length", "petal_width"]]
    except (KeyError, TypeError, ValueError) as e:
        return jsonify({"error": str(e)}), 422
    arr = np.array([features])
    proba = model.predict_proba(arr)[0]
    idx = int(np.argmax(proba))
    return jsonify({"prediction": CLASS_NAMES[idx], "confidence": round(float(proba[idx]), 4)})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)
'''
with open(os.path.join(EXERCISE_DIR, "app.py"), "w") as f:
    f.write(app_code)

# SOLUTION 4d: Write README.txt
readme = """Iris Classifier — Docker Deployment
====================================
Build the image:
  docker build -t iris-api:1.0 .

Run the container:
  docker run -p 8000:8000 iris-api:1.0

Test the API:
  curl -X POST http://localhost:8000/predict \\
       -H 'Content-Type: application/json' \\
       -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'

Health check:
  curl http://localhost:8000/health
"""
with open(os.path.join(EXERCISE_DIR, "README.txt"), "w") as f:
    f.write(readme)

# --- Verify all 4 files exist ---
expected_files = ["Dockerfile", "requirements.txt", "app.py", "README.txt"]
print("Deployment bundle contents:")
for filename in expected_files:
    path = os.path.join(EXERCISE_DIR, filename)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  [{'OK' if exists else 'MISSING'}] {filename} ({size} bytes)")
    assert exists, f"{filename} was not written to {EXERCISE_DIR}"

print(f"\nAll deployment files written to: {EXERCISE_DIR}")
print("Task 4 PASSED")

In [ ]:
# --- Final summary ---
print("=" * 55)
print("UNIT 4 LAB — FINAL GATE")
print("=" * 55)

gate = {
    "Task 1: Dockerfile generator produces valid content": df1 is not None and "FROM" in df1,
    "Task 2: Parser counts instructions correctly": counts1.get("FROM") == 1 if counts1 else False,
    "Task 3: Isolation simulation detects port conflicts": 8001 in (conflicts_bad or []),
    "Task 4: All 4 deployment files written": all(
        os.path.exists(os.path.join(EXERCISE_DIR, f))
        for f in ["Dockerfile", "requirements.txt", "app.py", "README.txt"]
    ),
}
for task, ok in gate.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)